In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_13_prng_and_statistical_tests.yaml')
ws.start()


$\def\AA{\text{Å}}$

# MSE 485 · Lesson 13: Pseudo-Random Number Generators, Marsaglia Hyperplanes & Chi-Squared Testing

_Session 45cc3380 · Estimated time: ~110 minutes · 40 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Distinguish True Random, Pseudo-Random (PRNG), and Quasi-Random (QRNG) sequences, comparing their reproducibility, serial correlations, and Monte Carlo integration error scaling.
- Analyze a PRNG as a deterministic finite-state automaton (State S, Transition T, Output Mapping F) and calculate period exhaustion times on modern GHz and massively parallel computers.
- Trace Linear Congruential Generators I_{n+1} = (a I_n + c) mod m by hand, diagnose bit-level periodicity in power-of-2 moduli, and calculate Marsaglia hyperplane bounds in d dimensions.
- Contrast Lagged Fibonacci Generators (LFGs) and Combined Multiple Recursive Generators against LCGs in terms of internal state size, period length, and seeding requirements.
- Apply Pearson's chi-squared test and the incomplete gamma tail probability Q(chi^2 | N-1) to evaluate empirical uniformity, interpreting both small-Q (biased) and large-Q (over-uniform) rejection tails.
- Design scalable parallel Monte Carlo random number streams using SPRNG, avoiding identical-seed cloning disasters and comparing Cycle Division (Leapfrogging) against Cycle and Iteration Parameterization.


In [2]:
ws.render_section('sec_rng_taxonomy_and_automata')

## 1. What Is a Random Number on a Computer? TRNG vs. PRNG vs. QRNG

As John von Neumann famously quipped in 1951, *"Anyone who considers arithmetical methods of producing random digits is, of course, in a state of sin"*—and Donald Knuth added in 1986, *"Random numbers should not be chosen at random."* A **single number** cannot be random; randomness is a **negative property** (the absence of predictable order) of an **infinite sequence** such that all subsequences are equally distributed and no intelligent gambler can profit by betting on the next draw.

In scientific computing we distinguish three fundamentally different families of generators:

| Generator Class | Mechanism | Reproducible? | Serial Correlation | $D$-Dim Quadrature Error | Primary Use Case |
| :--- | :--- | :---: | :--- | :--- | :--- |
| **True Random (TRNG)** | Physical noise (circuit thermal noise, Geiger decay counts, clock jitter) | **No** | Uncorrelated (if unbiased) | $\mathcal{O}(N^{-1/2})$ | Cryptographic keys, one-time seeds (too slow for MC: we need $\sim 10^{10}\text{ RN/s}$) |
| **Pseudo-Random (PRNG)** | Deterministic finite-state recurrence with long repeat period $L$ | **Yes** (from seed $S_0$) | Designed to vanish: $\langle f_1(x_{i+1})\cdots f_k(x_{i+k})\rangle = \langle f_1\rangle\cdots\langle f_k\rangle$ | $\mathcal{O}(N^{-1/2})$ | Markov Chain Monte Carlo, Molecular Dynamics, KMC |
| **Quasi-Random (QRNG)** | Low-discrepancy deterministic sequences (Halton, Sobol, van der Corput) | **Yes** | **Strongly negatively correlated** (sequentially fills the largest "holes" in space) | $\mathcal{O}\!\left(\frac{(\ln N)^D}{N}\right)$ | Low-dimensional ($D \lesssim 6$) numerical integration |

### The Finite-State Automaton Model & Cycle Exhaustion
Every computer PRNG is a deterministic finite-state machine defined by $(S, T, F)$:
1. **State $S_n$ (and Initial Seed $S_0$):** An internal memory buffer of $M$ bits ($2^M$ total possible states).
2. **Iteration Process $T$:** A deterministic transition rule $S_{n+1} = T(S_n)$.
3. **Output Mapping $F$:** Extracts an integer $I_n \in \{0, 1, \dots, 2^N-1\}$ or uniform real $U_n = I_n / 2^N \in [0, 1)$ from $S_n$.

Because the state space has at most $2^M$ distinct configurations and $T$ is deterministic, the sequence **must eventually repeat**, entering a periodic cycle of length $L \le 2^M$. If $T$ is a one-to-one (bijective) mapping, the state space partitions into one or more disjoint closed cycles.
- On a modern processor generating $\sim 10^9 \approx 2^{30}\text{ RN/s}$ per core:
  - **31-bit `rand` ($L \approx 2^{31}$):** Exhausted in **$\sim 1\text{–}2\text{ seconds}$** on a single processor!
  - **48-bit `drand48` ($L = 2^{48} \approx 2.8 \times 10^{14}$):** Exhausted in **$\sim 3\text{ days}$** on $1$ processor, or **$\sim 45\text{ minutes}$** across $100$ processors.
  - **64-bit LCG ($L \approx 2^{64} \approx 1.8 \times 10^{19}$):** Exhausted in $\sim 600\text{ years}$ on 1 core (or $\sim 2\text{ days}$ on $10^5$ cores).
  - **SPRNG Lagged Fibonacci ($L \approx 2^{1310}$):** Takes **$\sim 10^{375}\text{ years}$** even on $10^5$ processors!

### 1D Uniformity Moments
For a continuous uniform random variable $R \sim U(0, 1)$ with PDF $p(R) = 1$ on $[0, 1)$:
$$\langle R \rangle = \int_0^1 R\,dR = \frac{1}{2}, \qquad \sigma_R^2 = \left\langle (R - \langle R \rangle)^2 \right\rangle = \int_0^1 \left(R - \frac{1}{2}\right)^2 dR = \frac{1}{12} \approx 0.08333$$
In a finite sample of $N$ draws, the expected count inside a sub-interval $(a, b) \subset [0, 1]$ is $N(b - a) \pm \sqrt{N(b - a)[1 - (b - a)]} \approx N(b - a) \pm [N(b - a)]^{1/2}$.


In [3]:
ws.render_section('concept_prng_vs_qrng')

### Interactive Lab: Pseudo-Random Clumping vs. Quasi-Random Low-Discrepancy Sequences

Compare a standard **Pseudo-Random Number Generator (PRNG)** against a 2D **Halton Quasi-Random Number Generator (QRNG)** constructed from base-2 and base-3 van der Corput radical-inverse sequences. Switch the display mode to inspect 2D spatial voids/clumps on an $8\times 8$ grid or the log-log integration error convergence rate as $N$ grows.


**Key Takeaway:** Uncorrelated PRNG draws naturally clump together and leave $\mathcal{O}(\sqrt{n})$ Poisson voids, bounding Monte Carlo integration error to $\mathcal{O}(N^{-1/2})$. A QRNG deliberately violates serial independence so each new point lands inside the largest remaining void, accelerating low-dimensional integration convergence to $\mathcal{O}((\ln N)^D / N)$—but making QRNGs **unusable** for Markov Chain random walks where step-to-step independence is required!


In [4]:
ws.render_section('q13_01_rng_philosophy')

**Q1 · easy** · 💡

Why can a **single number** (such as $x = 0.42857$) never be classified as "random" on its own?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the lecture's definition of randomness as a property of an infinite sequence and its subsequences.

</div>
</details>

- [ ] **(A)** Because randomness is a property of an infinite sequence (the absence of predictable order and equal distribution across all subsequences), not of an isolated scalar value.
- [ ] **(B)** Because any floating-point number stored in binary on a computer is rational and therefore cannot be random.
- [ ] **(C)** Because a single number is only random if it is irrational, such as $\pi/4$ or $\sqrt{2}/2$.
- [ ] **(D)** Because randomness requires the number to be strictly equal to the ensemble mean $1/2$.

In [5]:
ws.render_section('q13_02_trng_limitations')

**Q2 · easy** · 💡

Select **all** reasons why physical **True Random Number Generators (TRNGs)** (e.g., thermal circuit noise or Geiger counter clicks) are generally **not** used directly to drive large-scale Monte Carlo or Molecular Dynamics simulations:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the three disadvantages of Truly Random numbers listed on slide 2 of `10.PRNG.pdf`.

</div>
</details>

- [ ] **(A)** They are not reproducible, making it impossible to replay an exact trajectory to debug a simulation crash.
- [ ] **(B)** They are far too slow and expensive when a simulation requires $\sim 10^{9}\text{–}10^{10}$ random numbers per second.
- [ ] **(C)** Raw physical noise sources often suffer from hardware drift, bias, or low statistical quality unless heavily post-processed.
- [ ] **(D)** Physical radioactive decay obeys a deterministic linear congruential recurrence with period $2^{16}$.

In [6]:
ws.render_section('q13_03_qrng_convergence')

**Q3 · medium** · 💡

How does a **Quasi-Random Number Generator (QRNG)** achieve an integration error convergence rate faster than $\mathcal{O}(N^{-1/2})$ (approaching $\mathcal{O}((\ln N)^D / N)$ in low dimension $D$), and what fundamental PRNG property does it sacrifice to do so?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about how QRNG places each subsequent point relative to previous points.

</div>
</details>

- [ ] **(A)** It sacrifices statistical independence (uncorrelatedness), introducing strong negative serial correlation so each new point sequentially fills the largest remaining 'holes' in space.
- [ ] **(B)** It sacrifices determinism, using quantum hardware interrupts to eliminate variance.
- [ ] **(C)** It sacrifices 1D uniformity, concentrating all points near the origin $x = 0$.
- [ ] **(D)** It uses a modulus $m = 16$ so that the sequence repeats every 16 evaluations.

In [7]:
ws.render_section('q13_04_automaton_components')

**Q4 · easy** · 💡

In the formal finite-state automaton model of a pseudo-random sequence shown in lecture, what are the roles of $S_n$, $T$, and $F$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Refer to the state-transition diagram $S_{n-1} \xrightarrow{T} S_n \xrightarrow{T} S_{n+1}$ with vertical arrows $F$.

</div>
</details>

- [ ] **(A)** $S_n$ is the internal state (initialized by the seed $S_0$), $T$ is the deterministic state-iteration process $S_{n+1} = T(S_n)$, and $F$ is the mapping from state $S_n$ to an output integer $I_n$ or uniform real $U_n$.
- [ ] **(B)** $S_n$ is the statistical variance, $T$ is the CPU temperature, and $F$ is the cumulative distribution function.
- [ ] **(C)** $S_n$ is the output floating-point number, $T$ is the chi-squared test, and $F$ is the Fourier transform.
- [ ] **(D)** $S_n$ is the modulus, $T$ is the multiplier, and $F$ is the additive constant.

In [8]:
ws.render_section('q13_05_state_bits_bound')

**Q5 · easy** · 💡

Suppose a PRNG maintains an internal state $S$ consisting of $M = 12$ bits. If the iteration mapping $T$ is one-to-one (bijective) and breaks the state space into $4$ disjoint cycles of equal length, what is the period (cycle length) $L$ of each cycle?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Total state space size is $2^M$. Divide $2^{12}$ by the number of equal-length cycles.

</div>
</details>

_Numeric answer required._

In [9]:
ws.render_section('q13_06_drand48_exhaustion')

**Q6 · medium** · 💡

The classic UNIX generator `drand48` is a 48-bit Linear Congruential Generator with maximum period $L = 2^{48} \approx 2.815 \times 10^{14}$. If you run a parallel Monte Carlo simulation across $P = 100$ processor cores, each consuming $10^9$ random numbers per second, how many **minutes** does it take for the entire simulation to exhaust $2^{48}$ random numbers? (Round to the nearest minute.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Divide $2^{48} \approx 2.81475 \times 10^{14}$ by the total consumption rate $(100 \times 10^9\text{ RN/s})$, then convert seconds to minutes.

</div>
</details>

_Numeric answer required._

In [10]:
ws.render_section('q13_07_ising_lattice_rn_budget')

**Q7 · easy** · 💡

Consider a 2D Ising Monte Carlo simulation on a $1024 \times 1024$ spin lattice ($\approx 10^6$ sites). To collect well-converged thermodynamic averages, you perform $10^7$ Monte Carlo sweeps (where $1$ sweep tests every spin once, consuming $1$ random number per spin). What is the base-10 logarithm $\log_{10}(N_{\text{total}})$ of the total number of random numbers required, and why does a 32-bit generator ($2^{32} \approx 4.29 \times 10^9$) fail? Enter $\log_{10}(N_{\text{total}})$.


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Multiply $10^6$ sites per sweep by $10^7$ sweeps, then take $\log_{10}$.

</div>
</details>

_Numeric answer required._

In [11]:
ws.render_section('q13_08_uniform_variance')

**Q8 · easy** · 💡

For an ideal continuous uniform random number generator producing $R \in [0, 1)$ with mean $\langle R \rangle = 0.5$, what is the exact theoretical variance $\sigma_R^2 = \langle (R - \langle R \rangle)^2 \rangle$? (Enter as a decimal to at least 4 significant figures.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $\int_0^1 (R - 1/2)^2 dR = 1/12$.

</div>
</details>

_Numeric answer required._

In [12]:
ws.render_section('q13_09_subinterval_fluctuation')

**Q9 · medium** · 💡

You generate $N = 10{,}000$ uniform random numbers $x \in [0, 1)$ and count how many fall inside the sub-interval $(a, b) = (0.20, 0.45)$. According to the lecture's quick Poisson uniformity rule $N(b-a) \pm [N(b-a)]^{1/2}$, what is the magnitude of the $1\sigma$ expected fluctuation $[N(b-a)]^{1/2}$ around the expected count of $2500$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\sqrt{N(b-a)}$ where $N = 10000$ and $b - a = 0.25$.

</div>
</details>

_Numeric answer required._

In [13]:
ws.render_section('q13_10_serial_uncorrelated_condition')

**Q10 · medium** · 💡

In a 3D Metropolis Monte Carlo simulation of $128$ atoms where each trial move picks a random atom index $i$ and a random 3D displacement $(\Delta x, \Delta y, \Delta z)$ (consuming $4$ random numbers per move, or $128 \times 4 = 512$ random numbers per pass), which mathematical condition guarantees that sequential random numbers do **not** introduce spurious multi-point correlations into the Markov chain?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the multi-point expectation value factorization on slides 3 and 11 of `10.PRNG.pdf`.

</div>
</details>

- [ ] **(A)** $\langle f_1(x_{i+1}) f_2(x_{i+2}) \cdots f_k(x_{i+k}) \rangle = \langle f_1(x_{i+1}) \rangle \langle f_2(x_{i+2}) \rangle \cdots \langle f_k(x_{i+k}) \rangle$ for arbitrary test functions $f_1, \dots, f_k$ and lags $k$.
- [ ] **(B)** $\langle x_{i+1} + x_{i+2} + \cdots + x_{i+k} \rangle = 0$ for all $k \ge 1$.
- [ ] **(C)** $x_{i+1} < x_{i+2} < \cdots < x_{i+k}$ for every block of $k$ numbers.
- [ ] **(D)** $\langle x_{i+1}^2 \rangle = \langle x_{i+1} \rangle^2$ so that the 1D variance vanishes.

In [14]:
ws.render_section('sec_lcg_marsaglia_and_lfg')

### Linear Congruential Generators (LCG)
The most widely analyzed PRNG is the **Linear Congruential Generator**, denoted $\text{LCG}(a, c, m, I_0)$:
$$I_{n+1} = (a I_n + c) \bmod m, \qquad R_n = \frac{I_n}{\text{float}(m)} \in [0, 1) \quad \left(\text{or } \frac{I_n}{m-1} \in [0, 1]\right)$$
where $a$ is the **multiplier**, $c$ is the **additive increment** ($c=0$ is a pure *multiplicative* LCG), $m$ is the **modulus**, and $I_0$ is the integer **seed**.

#### Complete Worked Trace: $\text{LCG}(a=5, c=1, m=16, I_0=0)$
Because $m = 16 = 2^4$ ($M = \log_2 16 = 4$ bits), the maximum possible period is $P = 16$. Starting from seed $I_0 = 0$:

| Step | Modular Arithmetic $(5 I_n + 1)\bmod 16$ | Integer | 4-Bit Binary | Real $R = I/16$ |
| :---: | :--- | :---: | :---: | :---: |
| $1$ | $5(0) + 1 = 1 - 0\times 16$ | **$1$** | `0001` | $0.0625$ |
| $2$ | $5(1) + 1 = 6 - 0\times 16$ | **$6$** | `0110` | $0.3750$ |
| $3$ | $5(6) + 1 = 31 - 1\times 16$ | **$15$** | `1111` | $0.9375$ |
| $4$ | $5(15) + 1 = 76 - 4\times 16$ | **$12$** | `1100` | $0.7500$ |
| $5$ | $5(12) + 1 = 61 - 3\times 16$ | **$13$** | `1101` | $0.8125$ |
| $6$ | $5(13) + 1 = 66 - 4\times 16$ | **$2$** | `0010` | $0.1250$ |
| $7\text{–}16$ | $\dots \to 11 \to 8 \to 9 \to 14 \to 7 \to 4 \to 5 \to 10 \to 3 \to 0$ | **$0$** | `0000` | $0.0000$ |

By the **Hull–Dobell Theorem**, $\text{LCG}(a, c, m)$ achieves the full period $L = m$ if and only if:
1. $c$ and $m$ are relatively prime ($\gcd(c, m) = 1$).
2. $a - 1$ is divisible by every prime factor of $m$.
3. If $m$ is a multiple of $4$, then $a - 1$ is also a multiple of $4$.

> [!WARNING]
> **Bit-Level Correlation in Power-of-2 LCGs ($m = 2^M$):** Look closely at the 4-bit binary column of $\text{LCG}(5, 1, 16, 0)$ above:
> - The **least significant bit (LSB)** strictly alternates `1, 0, 1, 0, 1, 0...` (period $2$) because $I_{n+1} \equiv (1\cdot I_n + 1)\bmod 2$.
> - In general, the $k$-th bit from the right depends *only* on the lowest $k$ bits and has a period of at most $2^k$. Even the most significant binary digit in small LCGs exhibits strong short-range pattern correlations!

### Marsaglia's Theorem (1968): "Random Numbers Fall Mainly in the Planes"
Even when an LCG has a full 1D period $L = m$, consecutive $d$-tuples $\mathbf{r}_n = (u_n, u_{n+1}, \dots, u_{n+d-1}) \in [0, 1)^d$ **do not fill $d$-dimensional space uniformly**. George Marsaglia proved that all $d$-tuples generated by an LCG lie on at most:
$$N_{\text{planes}} \le (d!\,m)^{1/d}$$
parallel $(d-1)$-dimensional hyperplanes!
- For a 32-bit LCG ($m = 2^{32}$) in $d = 3$, $(6 \times 2^{32})^{1/3} \approx 2953$ planes at best—and poorly chosen multipliers are far worse (IBM's infamous **RANDU** generator $I_{n+1} = 65539 I_n \bmod 2^{31}$ collapses onto just **15 parallel planes** in 3D!).
- **Practical Rule of Thumb:** For a $k$-bit LCG, **never group more than $k/2$ consecutive numbers together** in a high-dimensional vector.

### Beyond Simple LCGs: Lagged Fibonacci & Combined Generators
To overcome the short periods and hyperplane crystal lattices of single-word LCGs, modern libraries (like SPRNG) implement:
- **Additive / Multiplicative Lagged Fibonacci Generators (LFG):**
  $$z_n = (z_{n-p} \odot z_{n-q}) \bmod 2^m \qquad (\odot \in \{+, -, \times, \oplus\}, \quad p > q > 0)$$
  Because the internal state stores a **ring buffer of $p$ past $m$-bit words** ($M = p \times m$ total state bits), an LFG achieves colossal periods—up to $(2^p - 1)2^{m-1}$ for additive LFG and $(2^p - 1)2^{m-3}$ for multiplicative LFG—while requiring only one fast addition or multiplication per draw. Parallel streams are created by **varying the initial $p$-word seed table**.
- **Prime Modulus LCGs ($m = 2^{61}-1$ or $2^{31}-1$):** Eliminate power-of-2 low-bit periodicity.
- **Combined Multiple Recursive Generators (CMRG):** Combine a multi-lag recurrence $z_n = a_1 z_{n-1} + \dots + a_k z_{n-k}$ with an LCG to destroy hyperplane structure and extend the cycle length.


In [15]:
ws.render_section('concept_lcg_hyperplanes')

### Interactive Lab: LCG Cycle Traces & Marsaglia Hyperplanes in 2D Delay Space

Explore how the choice of multiplier $a$, increment $c$, and modulus $m$ controls both the **1D cycle length** $L$ and the **2D Marsaglia hyperplane geometry** $(u_n, u_{n+1})$. Compare the lecture's full-period $\text{LCG}(5, 1, 16)$ against degraded parameters and larger $m = 256$ generators where consecutive pairs crystallize onto parallel lines.


**Key Takeaway:** Even when an LCG visits every integer in $\{0, 1, \dots, m-1\}$ once per cycle (perfect 1D uniformity), plotting consecutive pairs $(u_n, u_{n+1})$ reveals that all points are trapped on a small number of parallel lines (**Marsaglia's planes**). Notice that changing $I_0$ in $\text{LCG}(5, 2, 16)$ hops between two disjoint 8-state cycles (even integers vs. odd integers)!


In [16]:
ws.render_section('q13_11_lcg_trace_step')

**Q11 · easy** · 💡

In the lecture's worked example $\text{LCG}(a=5, c=1, m=16)$ starting from seed $I_0 = 0$, the first four generated integers are $1, 6, 15, 12$. Following the recurrence $I_{n+1} = (5 I_n + 1) \bmod 16$, what is the **6th** generated integer (i.e., the integer immediately after $13$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $(5 \times 13 + 1) \bmod 16 = 66 - 4 \times 16$.

</div>
</details>

_Numeric answer required._

In [17]:
ws.render_section('q13_12_lcg_float_mapping')

**Q12 · easy** · 💡

An LCG produces integers $I_n \in \{0, 1, 2, \dots, m-1\}$. How do the two standard floating-point mappings $R_n = I_n / \text{float}(m)$ and $R_n' = I_n / \text{float}(m - 1)$ differ in their output range?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute the minimum ($I_n = 0$) and maximum ($I_n = m - 1$) integer values into both expressions.

</div>
</details>

- [ ] **(A)** $R_n = I_n / m$ maps onto the half-open interval $[0, 1)$ (excluding $1.0$), whereas $R_n' = I_n / (m - 1)$ maps onto the closed interval $[0, 1]$ (including both $0.0$ and $1.0$).
- [ ] **(B)** $R_n = I_n / m$ maps onto $(0, 1]$ (excluding $0.0$), whereas $R_n' = I_n / (m - 1)$ maps onto $(-1, 1)$.
- [ ] **(C)** Both mappings produce identical floating-point values for all $m$.
- [ ] **(D)** $R_n = I_n / m$ can exceed $1.0$ whenever $I_n = m - 1$.

In [18]:
ws.render_section('q13_13_lcg_lsb_alternation')

**Q13 · medium** · 💡

Why does the least significant bit (LSB, i.e., $I_n \bmod 2$) of any full-period power-of-2 LCG ($m = 2^M$) strictly alternate `0, 1, 0, 1, 0, 1...` with a period of only $2$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Take the recurrence $I_{n+1} = (a I_n + c) \bmod 2^M$ modulo $2$, noting that a full-period generator must have odd $a$ and odd $c$.

</div>
</details>

- [ ] **(A)** Because $m = 2^M$ is even, full period requires odd $a$ and odd $c$, so modulo $2$ the recurrence reduces to $I_{n+1} \equiv (1 \cdot I_n + 1) \bmod 2$, which flips parity every step.
- [ ] **(B)** Because IEEE-754 floating-point numbers always round the lowest mantissa bit to alternating parity.
- [ ] **(C)** Because $a$ is always an even multiple of $4$ in a full-period LCG.
- [ ] **(D)** Because the modulus $m = 2^M$ is a prime number.

In [19]:
ws.render_section('q13_14_hull_dobell_even_c')

**Q14 · medium** · 💡

Suppose you modify $\text{LCG}(5, 1, 16)$ by changing the additive constant from $c = 1$ to $c = 2$, giving $I_{n+1} = (5 I_n + 2) \bmod 16$. If you initialize with seed $I_0 = 0$, what is the cycle period $L$ of the resulting sequence? (Check with the interactive LCG preset `"LCG(5, 2, 16)"` if needed!)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

When $I_0 = 0$, $a = 5$, and $c = 2$, can $I_n$ ever become an odd integer?

</div>
</details>

_Numeric answer required._

In [20]:
ws.render_section('q13_15_marsaglia_3d_bound')

**Q15 · medium** · 💡

According to **Marsaglia's Theorem**, consecutive $d$-tuples $(u_n, u_{n+1}, \dots, u_{n+d-1})$ from an LCG with modulus $m$ fall on at most $N_{\text{planes}} \le (d!\,m)^{1/d}$ parallel hyperplanes. For a 24-bit LCG ($m = 2^{24} = 16{,}777{,}216$) used to generate 3D coordinates ($d = 3$), what is the theoretical maximum number of parallel 2D planes $(3!\,m)^{1/3}$ that can contain all generated 3D triplets? (Round to the nearest integer.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $(6 \times 2^{24})^{1/3} = 6^{1/3} \times 2^8 = 256 \times 6^{1/3}$.

</div>
</details>

_Numeric answer required._

In [21]:
ws.render_section('q13_16_k_over_2_rule')

**Q16 · easy** · 💡

Based on the lecture's rule of thumb for avoiding severe Marsaglia hyperplane correlations (*"For a $k$-bit generator, do not use them more than $k/2$ together"*), what is the maximum dimension $d_{\max}$ of a vector that should ever be constructed from consecutive draws of a **48-bit LCG** (such as `drand48`)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $k = 48$ into $k / 2$.

</div>
</details>

_Numeric answer required._

In [22]:
ws.render_section('q13_17_lfg_state_space')

**Q17 · medium** · 💡

How does an additive **Lagged Fibonacci Generator (LFG)** $z_n = (z_{n-k} + z_{n-l}) \bmod 2^m$ (with $k > l > 0$, e.g., $k = 1279, m = 32$) achieve an astronomical period ($L \sim 10^{375}\text{ years}$) even though its output word size is only $m = 32$ or $64$ bits?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What is the total number of bits $M$ in the internal state $S_n$ of a lagged recurrence with maximum lag $k$?

</div>
</details>

- [ ] **(A)** Its internal state stores a ring buffer of the last $k$ integers ($M = k \times m$ total state bits), expanding the state space to $2^{km}$ configurations instead of $2^m$.
- [ ] **(B)** It changes the modulus $2^m$ dynamically at every step using a hardware Geiger counter.
- [ ] **(C)** It divides each integer by zero to create an infinite floating-point mantissa.
- [ ] **(D)** It evaluates a transcendental cosine function $\cos(z_{n-1})$ at 1024-bit quadruple precision.

In [23]:
ws.render_section('q13_18_lfg_trace_step')

**Q18 · easy** · 💡

Consider a toy additive Lagged Fibonacci Generator $z_n = (z_{n-3} + z_{n-1}) \bmod 16$ ($k=3, l=1, m=4$). If the initial 3-word ring buffer is seeded with $(z_0, z_1, z_2) = (3, 7, 11)$, what is the value of $z_4$ (the **second** newly generated integer after $z_3$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute $z_3 = (z_0 + z_2) \bmod 16 = (3 + 11) \bmod 16 = 14$. Then compute $z_4 = (z_1 + z_3) \bmod 16$.

</div>
</details>

_Numeric answer required._

In [24]:
ws.render_section('q13_19_generator_parallelization_table')

**Q19 · medium** · 💡

According to the lecture's *"Common PRNG Generators"* summary table (slide 6 of `10.PRNG.pdf`), how are parallel independent streams parameterized in **Lagged Fibonacci** generators versus **48-bit / 64-bit / Prime-Modulus LCGs**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the 'Parallelization' column on slide 6: what is varied for Lagged Fibonacci vs. LCGs?

</div>
</details>

- [ ] **(A)** Lagged Fibonacci generators vary the initial seed table ('vary initialization' into disjoint cycles), whereas LCGs vary the additive prime constant $p$ (or multiplier $a$ for prime-modulus LCGs).
- [ ] **(B)** Lagged Fibonacci generators vary the modulus $2^m$ on each processor, whereas LCGs always use identical seeds and identical parameters on all processors.
- [ ] **(C)** Both Lagged Fibonacci and LCGs can only be parallelized by having Processor 0 broadcast every single random number over MPI.
- [ ] **(D)** Lagged Fibonacci generators vary the lag $k$ from $1$ to $P$, whereas LCGs vary the floating-point exponent.

In [25]:
ws.render_section('q13_20_lcg_64bit_period')

**Q20 · easy** · 💡

Approximately what is the maximum cycle length $L$ of a full-period **64-bit LCG** (or two combined 32-bit LCGs)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall $2^{10} \approx 10^3$, so $2^{64} = 16 \times (2^{10})^6$.

</div>
</details>

- [ ] **(A)** $L = 2^{64} \approx 1.8 \times 10^{19}$ (on the order of $10^{18}\text{–}10^{19}$)
- [ ] **(B)** $L = 64^2 = 4096$
- [ ] **(C)** $L = 2^{31} \approx 2.1 \times 10^9$
- [ ] **(D)** $L = 10^{64}$

In [26]:
ws.render_section('sec_chi_squared_testing')

## 3. Empirical Statistical Testing & Pearson's Chi-Squared Test

How do we test whether a sequence of $P$ draws (or $P$ $d$-dimensional tuples) is genuinely distributed according to a hypothesized probability distribution?

### Origin of the $\chi^2$ Distribution
Suppose $\{x_1, x_2, \dots, x_\nu\}$ are $\nu$ independent standard Gaussian random variables with mean $0$ and variance $1$. The sum of their squares:
$$y = \sum_{k=1}^\nu x_k^2$$
obeys the **chi-squared distribution** with $\nu$ degrees of freedom. Because $\langle x_k^2 \rangle = 1$ and $\text{Var}(x_k^2) = \langle x_k^4 \rangle - \langle x_k^2 \rangle^2 = 3 - 1 = 2$, the exact mean and variance of $y \sim \chi^2_\nu$ are:
$$\langle \chi^2 \rangle = \nu, \qquad \text{Var}(\chi^2) = 2\nu \implies \sigma_{\chi^2} = \sqrt{2\nu}$$
Roughly speaking, a valid random sample should yield a $\chi^2$ statistic inside the $\pm 1\sigma$ interval:
$$\nu - \sqrt{2\nu} \lesssim \chi^2 \lesssim \nu + \sqrt{2\nu}$$

### Pearson's $\chi^2$ Statistic for Binned Counts
To test a PRNG in $d$ dimensions, we partition the hypercube $[0, 1)^d$ into $N$ bins of known probabilities $p_i$ ($\sum_{i=1}^N p_i = 1$) and draw $P$ independent $d$-tuples.
- **Expected count in bin $i$:** $n_i = P p_i$ (for equal-volume bins, $n_i = P / N$).
- **Observed integer count in bin $i$:** $N_i$, with expected Poisson variance $\sigma_i^2 \approx n_i$.
- **Chi-squared statistic:**
  $$\chi^2 = \sum_{i=1}^N \frac{(N_i - n_i)^2}{n_i}$$
  *(Convention: omit terms where $n_i = N_i = 0$; any bin with $n_i = 0$ and $N_i \ne 0$ gives $\chi^2 = \infty$.)*
- **Degrees of Freedom ($\nu = N - 1$):** Because the observed counts obey the **sum rule** $\sum_{i=1}^N N_i = P = \sum_{i=1}^N n_i$, the last bin count $N_N$ is completely fixed once $N_1, \dots, N_{N-1}$ are known. Therefore, there are $\nu = N - 1$ degrees of freedom!

### Interpreting the Chi-Squared Tail Probability $Q(\chi^2 | N - 1)$
We quantify the test using the complementary cumulative distribution (the incomplete gamma function `gammaincc((N-1)/2, chi2/2)`):
$$Q(\chi^2 | \nu) = \text{Prob}(y > \chi^2)$$
which gives the probability that a genuinely random sequence would produce a $\chi^2$ value *larger* than the one observed.

| Tail Probability $Q(\chi^2 | N-1)$ | Interpretation | Physical Meaning |
| :---: | :---: | :--- |
| **$< 1\%$** ($Q < 0.01$) | **Reject** | $\chi^2 \gg \nu$: Fluctuations are way too large (biased, lumpy, or correlated generator) |
| **$1\%\text{–}5\%$** | **Suspect** | Moderately large $\chi^2$; investigate further |
| **$5\%\text{–}10\%$** | **Almost Suspect** | Borderline high $\chi^2$ |
| **$10\%\text{–}90\%$** | **Pass** | $\chi^2 \approx \nu \pm \sqrt{2\nu}$: Normal random Poisson fluctuations |
| **$90\%\text{–}95\%$** | **Almost Suspect** | Borderline low $\chi^2$ |
| **$95\%\text{–}99\%$** | **Suspect** | Fluctuations are unnaturally small |
| **$> 99\%$** ($Q > 0.99$) | **Reject** | $\chi^2 \ll \nu$: Counts $N_i$ are **too close** to $n_i$ (quasi-random, grid-like, or "cooked" data!) |

> [!IMPORTANT]
> **Passing Statistical Tests Is Necessary, Not Sufficient!** No universal battery of statistical tests can prove a PRNG is flawless for *every* physics algorithm (for instance, the Ferrenberg–Landau–Wong crisis where a "good" shift-register RNG biased the 2D Ising Wolff cluster algorithm). For careful research, **always rerun your simulation with several different generator families** (in SPRNG, simply relink or change the generator parameter) and combine the independent runs to reduce error bars!


In [27]:
ws.render_section('concept_chi_squared_lab')

### Interactive Lab: Pearson's Chi-Squared Test & Two-Sided Tail Rejection

Test four different sequences by binning $P$ draws into $N$ equal intervals on $[0, 1)$. Observe how a **genuine PRNG** fluctuates naturally inside the $\pm\sqrt{n_i}$ band ($10\% \le Q \le 90\%$), how a **biased** or **short-cycle** generator fails at the $Q < 1\%$ tail ($\chi^2 \gg N-1$), and how a **Quasi-Random (QRNG)** sequence fails at the **$Q > 99\%$ tail** ($\chi^2 \ll N-1$) because its counts are *too uniform*!


**Key Takeaway:** A $\chi^2$ value near zero ($\chi^2 \ll N-1$, giving $Q(\chi^2|N-1) > 99\%$) is **not** a "super-good" random number generator—it is a **rejection**! Truly independent random draws *must* exhibit $\pm\sqrt{n_i}$ Poisson noise from bin to bin; an unnaturally flat histogram proves strong negative correlations (like a quasi-random sequence or an LCG cycled an exact integer multiple of its period).


In [28]:
ws.render_section('q13_21_chi2_mean_variance')

**Q21 · easy** · 💡

If $y = \sum_{k=1}^{50} x_k^2$ is the sum of squares of $\nu = 50$ independent standard normal variables $x_k \sim \mathcal{N}(0, 1)$, its distribution is $\chi^2_\nu$ with mean $\langle y \rangle = \nu = 50$. What is the **standard deviation** $\sigma_y = \sqrt{\text{Var}(y)}$ of $y$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall $\text{Var}(y) = 2\nu$, so $\sigma_y = \sqrt{2\nu}$.

</div>
</details>

_Numeric answer required._

In [29]:
ws.render_section('q13_22_chi2_degrees_of_freedom')

**Q22 · easy** · 💡

When you divide a 3D unit cube $[0, 1)^3$ into $N = 100$ equal-volume bins and bin $P = 1000$ independent random triplets, why do you evaluate the chi-squared tail probability $Q(\chi^2 | \nu)$ at $\nu = N - 1 = 99$ degrees of freedom rather than $N = 100$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What constraint ('sum rule') links the observed bin counts $N_1, N_2, \dots, N_N$?

</div>
</details>

- [ ] **(A)** Because of the sum rule $\sum_{i=1}^N N_i = P$: once the counts in $N - 1$ bins are known, the count in the final bin is completely determined.
- [ ] **(B)** Because the first bin $i = 1$ always contains the seed $I_0$ and must be discarded.
- [ ] **(C)** Because $N - 1$ is always a prime number when $N = 100$.
- [ ] **(D)** Because the incomplete gamma function is undefined for even integers.

In [30]:
ws.render_section('q13_23_chi2_calculation')

**Q23 · medium** · 💡

You test a PRNG by drawing $P = 400$ numbers into $N = 4$ equal bins on $[0, 1)$, so the expected count in each bin is $n_i = 100$. If the observed integer counts in the 4 bins are $N_1 = 90$, $N_2 = 110$, $N_3 = 105$, and $N_4 = 95$, what is the exact value of Pearson's $\chi^2$ statistic?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\sum_{i=1}^4 (N_i - 100)^2 / 100$.

</div>
</details>

_Numeric answer required._

In [31]:
ws.render_section('q13_24_chi2_high_q_rejection')

**Q24 · medium** · 💡

Suppose a student tests a candidate random number generator across $N = 101$ bins ($\nu = 100$) and obtains $\chi^2 = 42.0$, corresponding to a tail probability $Q(\chi^2 | 100) = 0.9999998$ ($> 99\%$). The student concludes: *"Since $\chi^2$ measures error from the ideal uniform expectation, my generator is extra-uniform and passes with flying colors!"* Why is the student's conclusion **wrong**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the bottom row ($>99\%$) of the Numerical Recipes interpretation table on slide 16 of `10.PRNG.pdf`, or select the Quasi-Random sequence in the interactive Chi-Squared lab.

</div>
</details>

- [ ] **(A)** $Q > 99\%$ means $\chi^2$ is impossibly small ($\chi^2 \ll \nu$): genuine independent random numbers must fluctuate by $\sim \pm\sqrt{n_i}$ per bin, so $Q > 99\%$ rejects the generator as artificially flat or negatively correlated (e.g., quasi-random).
- [ ] **(B)** $Q$ represents the probability of failure, so $Q$ must always be less than $0.001$ for a good generator.
- [ ] **(C)** A value of $\chi^2 = 42.0$ is much larger than the expected mean $\nu = 100$.
- [ ] **(D)** The chi-squared statistic can never be smaller than $\nu = 100$ for any sequence of numbers.

In [32]:
ws.render_section('q13_25_chi2_interpretation_table')

**Q25 · easy** · 💡

According to the Numerical Recipes $\chi^2$ probability interpretation table presented in lecture, which range of tail probabilities $Q(\chi^2 | N - 1)$ is classified as an unambiguous **"pass"**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the green center row of the probability table on slide 16 of `10.PRNG.pdf`.

</div>
</details>

- [ ] **(A)** $10\% \le Q \le 90\%$ ($0.10 \le Q \le 0.90$)
- [ ] **(B)** $Q < 1\%$ ($Q < 0.01$)
- [ ] **(C)** $Q > 99\%$ ($Q > 0.99$)
- [ ] **(D)** $1\% \le Q \le 5\%$ ($0.01 \le Q \le 0.05$)

In [33]:
ws.render_section('q13_26_chi2_zero_expected_bin')

**Q26 · medium** · 💡

When computing $\chi^2 = \sum_i (N_i - n_i)^2 / n_i$ for a non-uniform distribution where some bin $k$ has theoretical expected count $n_k = 0$, how should that bin be handled?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the two bullet rules on slide 16 of `10.PRNG.pdf` regarding $n_i = 0$.

</div>
</details>

- [ ] **(A)** If $n_k = 0$ and observed $N_k = 0$, omit the term from the sum; if $n_k = 0$ and observed $N_k \ne 0$, the statistic correctly diverges to $\chi^2 = \infty$ (immediate rejection).
- [ ] **(B)** Always replace $n_k = 0$ with $n_k = 100$ so the generator automatically passes.
- [ ] **(C)** Set $\chi^2 = 0$ whenever any bin has $n_k = 0$, even if $N_k = 500$ samples landed in the forbidden bin.
- [ ] **(D)** Subtract $N_k$ from the degrees of freedom $\nu$.

In [34]:
ws.render_section('q13_27_hw_expected_fluctuation')

**Q27 · easy** · 💡

In the lecture's homework example (slide 15 of `10.PRNG.pdf`), you divide a cube into $N = 100$ equal bins and sample $P = 1000$ triplets. The expected count per bin is $n = P/N = 10$. What is the expected standard deviation $\sigma_n = (P/N)^{1/2}$ of the count in a single bin? (Enter to 2 decimal places.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\sqrt{10}$.

</div>
</details>

_Numeric answer required._

In [35]:
ws.render_section('q13_28_multiple_generators_practice')

**Q28 · medium** · 💡

Why does the lecture recommend **rerunning careful Monte Carlo simulations with several different PRNG algorithms** (e.g., comparing a 64-bit LCG against a Lagged Fibonacci generator in SPRNG)? Select **all** true statements:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review slide 17 ('Recommendation: For careful work use several generators!').

</div>
</details>

- [ ] **(A)** Passing standard empirical statistical tests is necessary but not sufficient; a specific physical MC algorithm may still resonate with subtle higher-order correlations of a particular generator.
- [ ] **(B)** The computational effort of running with multiple valid generators is not wasted, because their independent results can be combined to lower the final statistical error bars.
- [ ] **(C)** In modular libraries like SPRNG, switching to an entirely different generator family requires only relinking or changing a single initialization flag.
- [ ] **(D)** Averaging two simulations that both use the exact same broken 8-bit LCG with the exact same seed reduces systematic bias to zero.

In [36]:
ws.render_section('sec_parallel_prng_and_sprng')

## 4. Parallel Monte Carlo & Parallel PRNG Strategies (SPRNG)

Monte Carlo looks *"embarrassingly parallel"* at first glance, motivating two simple workflow patterns:
- **Cloning:** Running the **same physical parameters** across $P$ processors with **different random number streams**, then averaging the independent trajectories to shrink error bars by $1/\sqrt{P}$.
  - *Fatal Bug Alert:* If you accidentally initialize all $P$ clones with the **exact same random seed**, every processor executes the exact same trajectory—yielding **zero statistical error reduction** and wasting $(P-1)/P$ of the supercomputer!
- **Scanning:** Running **different physical parameters** (e.g., different temperatures $T_p$ or densities $\rho_p$) on each processor.
  - *Subtle Bug Alert:* If every processor uses the **same random number stream** while scanning neighbouring parameter points, the points on your phase diagram will have **artificially correlated fluctuations** (staying in phase!).

### Four Parallel PRNG Decomposition Strategies
How do we equip $P$ processors (or dynamically forked particle-splitting processes) with **uncorrelated**, high-quality random streams without a communication bottleneck?

1. **Central Dealer / Master-Worker (Avoid!):** Having Processor $0$ generate all random numbers and send them to workers creates a severe communication bottleneck and makes execution non-deterministic if message arrival order varies.
2. **Cycle Division (Leapfrog or Partition):** Splits a single generator's cycle of length $L$ among $P$ processors:
   - **Leapfrogging:** Processor $p \in \{0, \dots, P-1\}$ takes every $P$-th number: $I_{p},\, I_{p+P},\, I_{p+2P}, \dots$. For an LCG $I_{n+1} = (a I_n + c)\bmod m$, stepping by $P$ is itself an LCG with effective multiplier $A = a^P \bmod m$:
     $$I_{n+P} = \left(a^P I_n + c\frac{a^P - 1}{a - 1}\right) \bmod m$$
   - **Cycle Partitioning (Block Jump-Ahead):** Processor $p$ starts at offset $p \lfloor L / P \rfloor$ along the cycle.
   - *Caution:* Cycle division can amplify **long-range inter-stream correlations** in LCGs and runs the risk of stream overlap if $L/P$ is exhausted!
3. **Cycle Parameterization:** When a generator's $2^M$ state space naturally decomposes into many **disjoint closed cycles** (such as a Lagged Fibonacci Generator), assign each parallel stream a **different disjoint cycle** by varying its initial seed table.
4. **Iteration Parameterization:** Assign each stream a **different recurrence function $T^{(p)}$** (for example, giving each processor a distinct prime additive constant $c_p$ in a 48-bit/64-bit LCG, or a distinct multiplier $a_p$ in a prime-modulus LCG).

### The SPRNG Library (Scalable Parallel Random Number Generators)
Developed at NCSA, **SPRNG** provides a unified C/C++/Fortran API for parallel PRNG streams:
- `init_sprng(streamnum, nstreams, seed, param)`: Initializes independent stream `streamnum` out of `nstreams` from a common global `seed` and generator parameter `param`.
- `sprng(stream)` / `isprng(stream)`: Returns the next uniform double in $[0, 1)$ or random integer from `stream`.
- `make_sprng_seed()`: Generates a synchronized new seed across MPI processes.
- `spawn_sprng(stream, nspawned, &newstreams)`: Dynamically forks `nspawned` guaranteed-independent child streams from an existing stream (essential for branching/particle-splitting Monte Carlo where particles fork new processes on the fly without global synchronization!).
- `pack_sprng` / `unpack_sprng`: Serializes stream state into a byte buffer for checkpoint/restart or MPI migration.


In [37]:
ws.render_section('q13_29_cloning_vs_scanning')

**Q29 · easy** · 💡

In the terminology of parallel Monte Carlo simulations (slide 18 of `10.PRNG.pdf`), what is the distinction between **Cloning** and **Scanning**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the two definitions under 'Two methods for easy parallel MC' on slide 18.

</div>
</details>

- [ ] **(A)** Cloning runs the same physical system parameters on multiple processors with different random number streams to reduce statistical error, whereas Scanning runs different physical parameters (e.g., density or temperature) on each processor.
- [ ] **(B)** Cloning copies the exact same random seed to every processor to produce identical outputs, whereas Scanning uses a central processor to deal random numbers one by one.
- [ ] **(C)** Cloning is only possible on shared-memory GPUs, whereas Scanning requires a Geiger counter.
- [ ] **(D)** Cloning partitions space-time domains, whereas Scanning forks new child particles.

In [38]:
ws.render_section('q13_30_identical_seed_disaster')

**Q30 · medium** · 💡

A researcher runs a "cloned" Monte Carlo simulation across $P = 64$ MPI ranks to measure the average energy of a liquid. A single rank running for 1 hour achieves a statistical error bar of $\sigma_1 = 0.080\text{ eV}$ (so an ideal independent 64-rank run would reduce the error bar by $\sqrt{64} = 8$ to $0.010\text{ eV}$). However, the researcher accidentally initializes every rank with the **exact same seed** `seed = 12345` and `streamnum = 0`. What is the true statistical error bar (in $\text{eV}$) of the 64-rank average?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If all 64 ranks generate the exact same trajectory $E^{(1)}(t) = E^{(2)}(t) = \dots = E^{(64)}(t)$, does averaging 64 identical numbers reduce the variance?

</div>
</details>

_Numeric answer required._

In [39]:
ws.render_section('q13_31_scanning_in_phase_bug')

**Q31 · medium** · 💡

What goes wrong in a parallel **Scanning** simulation (where Processor $p$ simulates temperature $T_p$) if every processor uses the **exact same random number stream** and the trajectories stay "in phase"?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check slide 20 ('Unlikely to lead to problems unless they stay in phase') and consider whether the points on your $E(T)$ curve will be statistically independent.

</div>
</details>

- [ ] **(A)** The stochastic fluctuations at different temperatures $T_p$ become correlated with one another (correlated noise across the curve), invalidating independent-point curve fitting and error estimates.
- [ ] **(B)** Note that correlated sampling can sometimes be used deliberately for small differences, but if you treat the scanned points as independent measurements in a fit or phase diagram while their random walks stay in phase, their errors are strongly correlated!
- [ ] **(C)** The cycle length of each processor is reduced by a factor of $2^{32}$.
- [ ] **(D)** The compiler refuses to compile MPI programs when two ranks share a seed.

In [40]:
ws.render_section('q13_32_leapfrog_multiplier')

**Q32 · hard** · 💡

Suppose you parallelize our toy generator $\text{LCG}(a=5, c=1, m=16)$ across $P = 2$ processors using the **Leapfrog** (Cycle Division) method, so Processor $0$ takes $I_0, I_2, I_4, \dots$ and Processor $1$ takes $I_1, I_3, I_5, \dots$. Since stepping twice in $\text{LCG}(a, c, m)$ obeys $I_{n+2} = (A I_n + C) \bmod 16$ with effective multiplier $A = a^2 \bmod 16$, what is the effective multiplier $A$ for each processor's leapfrogged stream?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $A = 5^2 \bmod 16 = 25 \bmod 16$.

</div>
</details>

_Numeric answer required._

In [41]:
ws.render_section('q13_33_parallel_methods_taxonomy')

**Q33 · medium** · 💡

Match the three parallel RNG construction strategies from slide 19 of `10.PRNG.pdf`—**Cycle Division**, **Cycle Parameterization**, and **Iteration Parameterization**—to their correct mathematical definitions:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review the diagram and definitions on slide 19 ('Parallelization of RNGs').

</div>
</details>

- [ ] **(A)** Cycle Division splits a single cycle via leapfrogging or block partitioning; Cycle Parameterization assigns each stream to a different disjoint cycle of the same state-transition graph; Iteration Parameterization assigns a different recurrence function $T^{(p)}$ (e.g., different $a$ or $c$) to each stream.
- [ ] **(B)** Cycle Division varies the multiplier $a$; Cycle Parameterization uses a single cycle with leapfrog step $P$; Iteration Parameterization divides the modulus $m$ by the processor rank.
- [ ] **(C)** All three names refer to initializing every processor with the identical seed $S_0 = 0$.
- [ ] **(D)** Cycle Division uses Geiger counters; Cycle Parameterization uses Halton sequences; Iteration Parameterization uses $\chi^2$ tables.

In [42]:
ws.render_section('q13_34_sprng_spawn')

**Q34 · medium** · 💡

In a branching Monte Carlo simulation (such as neutron transport or Diffusion Quantum Monte Carlo with **particle splitting**, where a particle dynamically forks off new child processes at unpredictable times), which SPRNG function creates new independent random number streams from an existing stream **without** requiring global communication across all processors?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the list of SPRNG functions on slide 13 and the 'Particle splitting' example on slide 20.

</div>
</details>

- [ ] **(A)** `spawn_sprng(int *stream, int nspawned, int ***newstreams)`
- [ ] **(B)** `free_sprng(int *stream)`
- [ ] **(C)** `print_sprng(int *stream)`
- [ ] **(D)** `unpack_sprng(char *buffer)`

In [43]:
ws.render_section('q13_35_sprng_init_args')

**Q35 · easy** · 💡

When initializing parallel streams in SPRNG via `init_sprng(int streamnum, int nstreams, int seed, int param)`, how should `streamnum` and `seed` be set across $P$ MPI processes to obtain $P$ reproducible, independent streams from a single global run seed?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each MPI rank needs its own unique stream index `streamnum`, while sharing the common global `seed`.

</div>
</details>

- [ ] **(A)** Pass the unique MPI rank $p \in \{0, 1, \dots, P-1\}$ as `streamnum`, total ranks $P$ as `nstreams`, and the same shared global `seed` on all ranks.
- [ ] **(B)** Pass `streamnum = 0` and the same `seed` on every MPI rank so all ranks generate identical sequences.
- [ ] **(C)** Pass `nstreams = 0` and `param = -1` on every rank.
- [ ] **(D)** Call `free_sprng` before calling `init_sprng`.

In [44]:
ws.render_section('q13_36_parallel_feynman_exhaustion')

**Q36 · medium** · 💡

Slide 20 highlights evaluating thousands of Feynman diagram integrals in parallel, each to an accuracy of $\epsilon = 10^{-6}$. Why is **cycle length exhaustion** a primary hazard in this calculation?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How many Monte Carlo samples $N$ are required to reach a relative statistical error $\epsilon \sim N^{-1/2} = 10^{-6}$ for just ONE integral?

</div>
</details>

- [ ] **(A)** Reaching $\epsilon \sim N^{-1/2} = 10^{-6}$ requires $N \sim 10^{12}$ samples per integral, so evaluating thousands of multi-dimensional integrals consumes $10^{15}\text{–}10^{16}$ random numbers—completely exhausting 32-bit and even 48-bit (`drand48`) generators!
- [ ] **(B)** Reaching $\epsilon = 10^{-6}$ requires only $N = 6$ random numbers, which underflows a 64-bit register.
- [ ] **(C)** Feynman diagrams can only be integrated using physical Geiger counters.
- [ ] **(D)** Cycle exhaustion only occurs when $N < 16$.

In [45]:
ws.render_section('q13_37_spacetime_partitioning_hazard')

**Q37 · medium** · 💡

In a **space-time domain-decomposed** parallel Monte Carlo simulation (where each spatial subdomain is assigned to a different processor with its own PRNG stream), why are inter-stream correlations between neighboring processors particularly dangerous?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What happens at the boundary interface between adjacent spatial subdomains?

</div>
</details>

- [ ] **(A)** Atoms or spins across the boundary between neighboring processors interact directly at the same time step; if their local PRNG streams are correlated, artificial synchronized forces or phase boundaries appear at processor interfaces.
- [ ] **(B)** Domain decomposition never uses random numbers inside the subdomains.
- [ ] **(C)** Correlated streams cause the spatial box length $L$ to shrink to zero.
- [ ] **(D)** Each processor only calls its PRNG once at $t = 0$.

In [46]:
ws.render_section('q13_38_lcg_mean_exact_cycle')

**Q38 · medium** · 💡

For our full-period generator $\text{LCG}(5, 1, 16, 0)$, the 16 generated integers over one complete cycle are $\{0, 1, 2, \dots, 15\}$, mapped to $R_n = I_n / 16 \in [0, 1)$. What is the exact discrete cycle average $\langle R \rangle_{\text{cycle}} = \frac{1}{16}\sum_{n=0}^{15} \frac{I_n}{16}$? (Enter as a decimal to 4 decimal places.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use $\sum_{k=0}^{15} k = \frac{15 \times 16}{2} = 120$, then divide by $16 \times 16 = 256$.

</div>
</details>

_Numeric answer required._

In [47]:
ws.render_section('q13_39_chi2_cycle_exhaustion_trap')

**Q39 · hard** · 💡

Suppose you take the full-period generator $\text{LCG}(5, 1, 16, 0)$ (period $m = 16$), draw $P = 1600$ numbers (exactly $100$ full cycles!), and bin them into $N = 16$ equal bins on $[0, 1)$. Because every integer $0, 1, \dots, 15$ occurs **exactly** $100$ times across $100$ full periods, every bin receives $N_i = 100$ counts (matching the expected count $n_i = 100$ with zero error!). What value of $\chi^2$ do you obtain, and what is the tail probability $Q(\chi^2 | 15)$? Enter the value of $Q(\chi^2 | 15)$.


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

When $N_i = n_i$ for every bin, $\chi^2 = 0$. What is $\text{Prob}(y > 0)$ for a continuous positive random variable $y \sim \chi^2_{15}$?

</div>
</details>

_Numeric answer required._

In [48]:
ws.render_section('q13_40_sprng_checkpoint_functions')

**Q40 · easy** · 💡

When running a multi-day parallel Monte Carlo simulation on a supercomputer with a 24-hour wall-clock queue limit, you must save the exact internal state of every processor's PRNG stream to disk so the restart job continues seamlessly without repeating or skipping random numbers. Which pair of SPRNG functions serializes and restores a stream's state?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the SPRNG function list on slide 13 of `10.PRNG.pdf`.

</div>
</details>

- [ ] **(A)** `pack_sprng(int *stream, char **buffer)` and `unpack_sprng(char *buffer)`
- [ ] **(B)** `make_sprng_seed()` and `free_sprng(int *stream)`
- [ ] **(C)** `sprng(int *stream)` and `isprng(int *stream)`
- [ ] **(D)** `print_sprng(int *stream)` and `spawn_sprng(...)`

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()